# Check Stimulus ID Against Recorded Audio Track

This notebook checks whether `stimulus_id` in the BIDS `events.tsv` files matches the actual WAV stimulus recorded in the audio/AUD channel.

It does not modify the BIDS dataset. It only writes QC reports.

Rationale: `value` is the raw EEG trigger code, while `stimulus_id` should identify the actual WAV segment. Because the Alice dataset has known trigger/stimulus alignment issues, we should verify `stimulus_id` against the recorded audio track before changing the public BIDS events.

In [1]:
from pathlib import Path
import json
import wave

import mne
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, hilbert, resample_poly

BIDS_ROOT = Path('/Users/yanyuwoo/Data/bids')
STIMULI_DIR = BIDS_ROOT / 'stimuli'
QC_DIR = Path('/Users/yanyuwoo/Data/Alice Comprehension/qc/stimulus_id_audio_alignment')
QC_DIR.mkdir(parents=True, exist_ok=True)

# Default: check every subject and every event row.
SUBJECTS_TO_CHECK = 'all'

# Compare each recorded audio segment against all WAV files. This is slower but actually tests the ID.
COMPARE_ALL_WAVS = True

# Envelope settings for robust audio-track comparison.
QC_FS = 50
ENVELOPE_LOWPASS_HZ = 8
MAX_LAG_SEC = 1.0
MIN_COMPARISON_SEC = 5.0

print(f'BIDS root: {BIDS_ROOT}')
print(f'QC dir: {QC_DIR}')
print(f'Subjects to check: {SUBJECTS_TO_CHECK}')
print(f'Compare all WAVs: {COMPARE_ALL_WAVS}')

BIDS root: /Users/yanyuwoo/Data/bids
QC dir: /Users/yanyuwoo/Data/Alice Comprehension/qc/stimulus_id_audio_alignment
Subjects to check: all
Compare all WAVs: True


## Helper Functions

The check compares amplitude envelopes rather than raw waveforms. This is more robust to scale differences and possible polarity differences between the original WAV and the recorded audio/AUD channel.

In [2]:
AUDIO_CHANNEL_CANDIDATES = {'aud', 'audio', 'aux', 'aux5', 'ox'}


def subject_from_path(path):
    return path.parts[-3]


def read_wav_mono(path):
    with wave.open(str(path), 'rb') as wav:
        fs = wav.getframerate()
        n_channels = wav.getnchannels()
        sampwidth = wav.getsampwidth()
        frames = wav.readframes(wav.getnframes())

    dtype_by_width = {1: np.uint8, 2: np.int16, 4: np.int32}
    if sampwidth not in dtype_by_width:
        raise ValueError(f'Unsupported WAV sample width {sampwidth}: {path}')

    data = np.frombuffer(frames, dtype=dtype_by_width[sampwidth]).astype(float)
    if sampwidth == 1:
        data = data - 128
    if n_channels > 1:
        data = data.reshape(-1, n_channels).mean(axis=1)
    return data, fs


def lowpass(data, fs, cutoff):
    if cutoff >= fs / 2:
        return data
    b, a = butter(4, cutoff / (fs / 2), btype='lowpass')
    return filtfilt(b, a, data)


def envelope_at_qc_fs(data, fs, qc_fs=QC_FS):
    data = np.asarray(data, dtype=float)
    data = data - np.nanmean(data)
    if np.nanstd(data) == 0:
        return np.array([])
    env = np.abs(hilbert(data))
    env = lowpass(env, fs, ENVELOPE_LOWPASS_HZ)
    env = resample_poly(env, qc_fs, int(round(fs)))
    env = env - np.nanmean(env)
    std = np.nanstd(env)
    if std == 0 or not np.isfinite(std):
        return np.array([])
    return env / std


def lagged_corr(x, y, max_lag_samples):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = min(len(x), len(y))
    if n < int(MIN_COMPARISON_SEC * QC_FS):
        return np.nan, np.nan
    x = x[:n]
    y = y[:n]

    best_r = np.nan
    best_lag = np.nan
    for lag in range(-max_lag_samples, max_lag_samples + 1):
        if lag < 0:
            xx = x[-lag:]
            yy = y[:len(xx)]
        elif lag > 0:
            yy = y[lag:]
            xx = x[:len(yy)]
        else:
            xx = x
            yy = y
        if len(xx) < int(MIN_COMPARISON_SEC * QC_FS):
            continue
        r = np.corrcoef(xx, yy)[0, 1]
        if np.isfinite(r) and (not np.isfinite(best_r) or r > best_r):
            best_r = float(r)
            best_lag = lag / QC_FS
    return best_r, best_lag


def find_audio_channel(raw):
    lower_to_name = {ch.lower(): ch for ch in raw.ch_names}
    for candidate in AUDIO_CHANNEL_CANDIDATES:
        if candidate in lower_to_name:
            return lower_to_name[candidate]
    for ch in raw.ch_names:
        lower = ch.lower()
        if any(token in lower for token in ['aud', 'audio', 'aux']):
            return ch
    return None


def load_raw_subject(subject):
    vhdr = BIDS_ROOT / subject / 'eeg' / f'{subject}_task-alice_eeg.vhdr'
    return mne.io.read_raw_brainvision(vhdr, preload=False, verbose='ERROR')

## Load WAV Envelopes

These are the candidate stimulus identities. If `stimulus_id` is correct, the recorded AUD segment should match the corresponding WAV better than the other WAV files.

In [3]:
wav_rows = []
wav_envelopes = {}
wav_durations = {}

for wav_path in sorted(STIMULI_DIR.glob('*.wav'), key=lambda p: int(p.stem)):
    stimulus_id = int(wav_path.stem)
    wav_data, wav_fs = read_wav_mono(wav_path)
    wav_env = envelope_at_qc_fs(wav_data, wav_fs)
    wav_envelopes[stimulus_id] = wav_env
    wav_durations[stimulus_id] = len(wav_data) / wav_fs
    wav_rows.append({
        'stimulus_id': stimulus_id,
        'wav_path': str(wav_path),
        'wav_fs': wav_fs,
        'duration_sec': wav_durations[stimulus_id],
        'envelope_samples_at_qc_fs': len(wav_env),
    })

wav_table = pd.DataFrame(wav_rows)
display(wav_table)
print(f'Loaded WAV envelopes: {sorted(wav_envelopes)}')

,stimulus_id,wav_path,wav_fs,duration_sec,envelope_samples_at_qc_fs
0,1,/Users/yanyuwoo/Data/bids/stimuli/1.wav,44100,57.540612,2878
1,2,/Users/yanyuwoo/Data/bids/stimuli/2.wav,44100,60.845193,3043
2,3,/Users/yanyuwoo/Data/bids/stimuli/3.wav,44100,63.259433,3163
3,4,/Users/yanyuwoo/Data/bids/stimuli/4.wav,44100,69.988571,3500
4,5,/Users/yanyuwoo/Data/bids/stimuli/5.wav,44100,66.272540,3314
5,6,/Users/yanyuwoo/Data/bids/stimuli/6.wav,44100,63.777551,3189
6,7,/Users/yanyuwoo/Data/bids/stimuli/7.wav,44100,62.896848,3145
7,8,/Users/yanyuwoo/Data/bids/stimuli/8.wav,44100,57.310612,2866
8,9,/Users/yanyuwoo/Data/bids/stimuli/9.wav,44100,57.226145,2862
9,10,/Users/yanyuwoo/Data/bids/stimuli/10.wav,44100,61.269660,3064


Loaded WAV envelopes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


## Run Alignment Check

This loops over subjects and event rows. It compares the recorded AUD envelope after each event onset against the expected WAV envelope and, when `COMPARE_ALL_WAVS=True`, against all 12 WAV envelopes.

In [4]:
all_subjects = sorted(
    [path.name for path in BIDS_ROOT.glob('sub-*') if path.is_dir()],
    key=lambda s: int(s.split('-')[1]),
)
subjects = all_subjects if SUBJECTS_TO_CHECK == 'all' else SUBJECTS_TO_CHECK

max_lag_samples = int(MAX_LAG_SEC * QC_FS)
event_rows = []
candidate_rows = []
subject_rows = []

for subject in subjects:
    events_path = BIDS_ROOT / subject / 'eeg' / f'{subject}_task-alice_events.tsv'
    if not events_path.exists():
        subject_rows.append({'subject': subject, 'status': 'missing_events'})
        continue

    events = pd.read_csv(events_path, sep='\t')
    if 'stimulus_id' not in events.columns:
        subject_rows.append({'subject': subject, 'status': 'missing_stimulus_id'})
        continue

    try:
        raw = load_raw_subject(subject)
        audio_ch = find_audio_channel(raw)
        if audio_ch is None:
            subject_rows.append({'subject': subject, 'status': 'missing_audio_channel'})
            continue

        raw_fs = raw.info['sfreq']
        audio_pick = raw.ch_names.index(audio_ch)
        ok_events = 0
        mismatch_events = 0

        for event_index, row in events.iterrows():
            expected_id = int(row['stimulus_id'])
            onset = float(row['onset'])
            duration = wav_durations.get(expected_id)
            if duration is None:
                event_rows.append({'subject': subject, 'event_index': event_index, 'status': 'missing_wav_for_stimulus_id'})
                continue

            start = int(round(onset * raw_fs))
            stop = int(round((onset + duration) * raw_fs))
            if start < 0 or stop > raw.n_times:
                event_rows.append({
                    'subject': subject,
                    'event_index': event_index,
                    'value': row.get('value'),
                    'stimulus_id': expected_id,
                    'status': 'audio_window_out_of_bounds',
                })
                continue

            recorded = raw.get_data(picks=[audio_pick], start=start, stop=stop, verbose='ERROR')[0]
            recorded_env = envelope_at_qc_fs(recorded, raw_fs)
            candidates = sorted(wav_envelopes) if COMPARE_ALL_WAVS else [expected_id]

            scores = []
            for candidate_id in candidates:
                r, lag_sec = lagged_corr(recorded_env, wav_envelopes[candidate_id], max_lag_samples)
                scores.append((candidate_id, r, lag_sec))
                candidate_rows.append({
                    'subject': subject,
                    'event_index': event_index,
                    'value': row.get('value'),
                    'stimulus_id': expected_id,
                    'candidate_wav_id': candidate_id,
                    'correlation': r,
                    'best_lag_sec': lag_sec,
                })

            valid_scores = [(cid, r, lag) for cid, r, lag in scores if np.isfinite(r)]
            if valid_scores:
                best_id, best_r, best_lag = max(valid_scores, key=lambda item: item[1])
                expected_scores = [item for item in valid_scores if item[0] == expected_id]
                expected_r, expected_lag = (expected_scores[0][1], expected_scores[0][2]) if expected_scores else (np.nan, np.nan)
                rank = 1 + sum(r > expected_r for _, r, _ in valid_scores if np.isfinite(expected_r))
                match = best_id == expected_id
                ok_events += int(match)
                mismatch_events += int(not match)
                status = 'ok' if match else 'best_wav_mismatch'
            else:
                best_id, best_r, best_lag = np.nan, np.nan, np.nan
                expected_r, expected_lag, rank = np.nan, np.nan, np.nan
                match = False
                mismatch_events += 1
                status = 'no_valid_correlation'

            event_rows.append({
                'subject': subject,
                'event_index': event_index,
                'onset': onset,
                'trial_type': row.get('trial_type'),
                'value': row.get('value'),
                'stimulus_id': expected_id,
                'best_matching_wav': best_id,
                'expected_r': expected_r,
                'expected_lag_sec': expected_lag,
                'best_r': best_r,
                'best_lag_sec': best_lag,
                'expected_rank': rank,
                'best_matches_stimulus_id': match,
                'audio_channel': audio_ch,
                'status': status,
            })

        subject_rows.append({
            'subject': subject,
            'audio_channel': audio_ch,
            'n_events': len(events),
            'ok_events': ok_events,
            'mismatch_events': mismatch_events,
            'status': 'ok' if mismatch_events == 0 else 'has_mismatch',
        })
    except Exception as exc:
        subject_rows.append({'subject': subject, 'status': 'failed', 'error': repr(exc)})

event_qc = pd.DataFrame(event_rows)
candidate_qc = pd.DataFrame(candidate_rows)
subject_qc = pd.DataFrame(subject_rows)

event_path = QC_DIR / 'stimulus_id_audio_alignment_events.csv'
candidate_path = QC_DIR / 'stimulus_id_audio_alignment_candidates.csv'
subject_path = QC_DIR / 'stimulus_id_audio_alignment_subjects.csv'

event_qc.to_csv(event_path, index=False)
candidate_qc.to_csv(candidate_path, index=False)
subject_qc.to_csv(subject_path, index=False)

print(f'Wrote event QC: {event_path}')
print(f'Wrote candidate QC: {candidate_path}')
print(f'Wrote subject QC: {subject_path}')
display(subject_qc)
display(event_qc)

Wrote event QC: /Users/yanyuwoo/Data/Alice Comprehension/qc/stimulus_id_audio_alignment/stimulus_id_audio_alignment_events.csv
Wrote candidate QC: /Users/yanyuwoo/Data/Alice Comprehension/qc/stimulus_id_audio_alignment/stimulus_id_audio_alignment_candidates.csv
Wrote subject QC: /Users/yanyuwoo/Data/Alice Comprehension/qc/stimulus_id_audio_alignment/stimulus_id_audio_alignment_subjects.csv


,subject,audio_channel,n_events,ok_events,mismatch_events,status
0,sub-01,AUD,12.0,12.0,0.0,ok
1,sub-02,AUD,11.0,11.0,0.0,ok
2,sub-03,AUD,12.0,11.0,1.0,has_mismatch
3,sub-04,AUD,12.0,12.0,0.0,ok
4,sub-05,NaN,NaN,NaN,NaN,missing_audio_channel
5,sub-06,AUD,12.0,12.0,0.0,ok
6,sub-07,AUD,12.0,12.0,0.0,ok
7,sub-08,AUD,12.0,12.0,0.0,ok
8,sub-09,AUD,12.0,0.0,12.0,has_mismatch
9,sub-10,AUD,12.0,12.0,0.0,ok


,subject,event_index,onset,trial_type,value,stimulus_id,best_matching_wav,expected_r,expected_lag_sec,best_r,best_lag_sec,expected_rank,best_matches_stimulus_id,audio_channel,status
0,sub-01,0,3.664,Stimulus/1,1,1,1,0.361117,-0.04,0.361117,-0.04,1,True,AUD,ok
1,sub-01,1,61.292,Stimulus/2,5,2,2,0.435430,-0.04,0.435430,-0.04,1,True,AUD,ok
2,sub-01,2,122.188,Stimulus/3,6,3,3,0.520779,-0.04,0.520779,-0.04,1,True,AUD,ok
3,sub-01,3,185.500,Stimulus/4,7,4,4,0.482766,-0.04,0.482766,-0.04,1,True,AUD,ok
4,sub-01,4,255.544,Stimulus/5,8,5,5,0.472915,-0.02,0.472915,-0.02,1,True,AUD,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,sub-49,6,444.298,Stimulus/8,7,8,8,0.595201,-0.04,0.595201,-0.04,1,True,AUD,ok
549,sub-49,7,501.678,Stimulus/9,8,9,9,0.613630,-0.04,0.613630,-0.04,1,True,AUD,ok
550,sub-49,8,558.956,Stimulus/10,9,10,10,0.593010,-0.04,0.593010,-0.04,1,True,AUD,ok
551,sub-49,9,620.284,Stimulus/11,10,11,11,0.662979,-0.04,0.662979,-0.04,1,True,AUD,ok


## Interpret Results

The key column is `best_matches_stimulus_id` in the event-level report.

- `True`: the recorded audio segment best matched the WAV file indicated by `stimulus_id`.
- `False`: another WAV file matched better, so the event should be inspected before changing the dataset.

A perfect or near-perfect check would show all valid events with `best_matches_stimulus_id=True`. If mismatches appear systematically, inspect `value`, `stimulus_id`, `best_matching_wav`, and `expected_lag_sec` before editing BIDS events.